In [1]:
%pip install uv
!uv pip install "git+https://github.com/ibm-granite-community/utils.git" transformers langchain_classic langchain_text_splitters langchain_huggingface sentence_transformers langchain_chroma chromadb "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" wget

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
Resolved 123 packages in 3.96s
Checked 123 packages in 41ms


In [2]:
%pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
import os

# Зареждане на модела за векторизиране (embeddings)
embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(model_name=embeddings_model_path)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

# Създаване на векторната база
vector_db = Chroma(embedding_function=embeddings_model)


# Свързване с безплатен езиков модел Llama 3 през Groq
model = ChatGroq(
    model="openai/gpt-oss-20b", 
    groq_api_key=os.environ.get('GROQ_API_KEY')
)
print("Моделите са заредени успешно!")

c:\Users\User\Desktop\rag project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 74/74 [00:00<00:00, 3797.28it/s]
This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.


Моделите са заредени успешно!


In [5]:
import wget
from langchain_core.document_loaders import Blob
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter

filename = 'state_of_the_union.txt'
url = 'https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/foundation_models/state_of_the_union.txt'

if not os.path.isfile(filename):
    wget.download(url, out=filename)

blob = Blob.from_path(filename)
document = Document(page_content=blob.as_string(), metadata={"source": filename})

text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embeddings_tokenizer,
    chunk_size=embeddings_tokenizer.max_len_single_sentence,
    chunk_overlap=0,
)
texts = text_splitter.split_documents([document])
print(f"Документът е разделен на {len(texts)} части.")

Документът е разделен на 2 части.


In [6]:
print("Записване в базата данни... Моля, изчакайте.")
ids = vector_db.add_documents(texts)
print(f"Готово! {len(ids)} документа са добавени във векторната база.")

Записване в базата данни... Моля, изчакайте.
Готово! 2 документа са добавени във векторната база.


In [7]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import textwrap

query = "What is the capital of Bulgaria?"

prompt_template = ChatPromptTemplate.from_template("""
You are an assistant that answers questions ONLY using the context below.

Rules:
- Use only information that is explicitly stated in the context.
- Do NOT use your own general knowledge, even if you know the answer.
- If the answer is not in the context, reply exactly:
  "I could not find this information in the provided documents."

<context>
{context}
</context>

Question: {input}
""")

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# диагностика: колко големи са парчетата
docs = retriever.invoke(query)
print("Дължини на парчетата:", [len(d.page_content) for d in docs])

def format_docs(docs, max_chars=2000):
    return "\n\n".join(d.page_content[:max_chars] for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt_template
    | model
    | StrOutputParser()
)

answer = rag_chain.invoke(query)
print("\n--- ОТГОВОР ---")
print(textwrap.fill(answer, width=80))

Дължини на парчетата: [35704, 2832]

--- ОТГОВОР ---
I could not find this information in the provided documents.
